# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

# IMPORTING

## Dataset

In [ ]:
import config
import pandas as pd

file_path = config.TIMBER_STOCK_PATH / 'complete_timber_small.csv'

# Try common combinations
read_attempts = [
    {"sep": ",", "encoding": "utf-8"},
    {"sep": ";", "encoding": "utf-8"},
    {"sep": ",", "encoding": "latin1"},
    {"sep": ";", "encoding": "latin1"},
]

df_input_stock = None
for opts in read_attempts:
    try:
        df_try = pd.read_csv(file_path, **opts)
        # Valid if we get more than 1 column
        if df_try.shape[1] > 1:
            df_input_stock = df_try
            print(f"✅ Loaded with sep='{opts['sep']}' and encoding='{opts['encoding']}'")
            break
    except Exception:
        pass

if df_input_stock is None:
    raise ValueError("Could not parse CSV with tested delimiter/encoding combinations.")

# Clean column names
df_input_stock.columns = df_input_stock.columns.str.strip()

print("Gevonden kolommen:", df_input_stock.columns.tolist())
print(f"\nDataset bevat {df_input_stock.shape[0]} elementen\n")
display(df_input_stock.head())

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [ ]:
import json

json_path = 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

# GEOMETRY

## Random geometry

In [ ]:
import json
import random
import c11_params

# ==========================================
# 1. RANDOM PARAMETERS GENEREREN (De "DNA" string)
# ==========================================
def get_random_design(json_path):
    """Leest de search space en trekt voor elke knop een willekeurige waarde."""
    with open(json_path, 'r') as f:
        search_space = json.load(f)

    random_params = {}

    for var_name, rules in search_space.items():
        if rules['type'] == 'continuous':
            # Kies een willekeurig kommagetal (bijv. voor u en v)
            random_params[var_name] = random.uniform(rules['min'], rules['max'])

        elif rules['type'] == 'discrete':
            # Kies exact één van de toegestane stapjes (bijv. voor shift_x)
            random_params[var_name] = random.choice(rules['options'])

    return random_params

# --- TEST DE FUNCTIE ---
print("Stap 1: Willekeurig DNA genereren...")
mijn_random_ontwerp = get_random_design(json_path)

print("\nSucces! Dit is het DNA van ons proef-ontwerp:")
for sleutel, waarde in mijn_random_ontwerp.items():
    print(f" - {sleutel}: \t{waarde:.3f}")

# Opzet

In [ ]:
import pandas as pd
from geometry import generate_sample_vertices
from reconstruction import reconstruct_edges

# --- UITVOEREN VAN STAP 2 ---
print("Stap 2: DNA vertalen naar 3D Geometrie...")

# We gebruiken 'mijn_random_ontwerp' uit de VORIGE cel!
vertices_list = generate_sample_vertices(sample_id=0, params=mijn_random_ontwerp)
df_vertices = pd.DataFrame(vertices_list)
df_edges = reconstruct_edges(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

print(f"✅ Succes! Geometrie opgebouwd met {len(df_vertices)} knooppunten en {len(df_edges)} staven.")
print("\nVoorbeeld van de gegenereerde Vertices (eerste 5):")
print(df_vertices.head())

## Quick 3D visual check

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Build a lookup so we can quickly fetch coordinates for each vertex id.
vertex_lookup = df_vertices.set_index("vertex_index")[["x", "y", "z", "layer"]].to_dict("index")

def _to_vertex_key(v):
    v_str = str(v)
    return v_str if v_str.startswith("v") else f"v{v_str}"

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

# Draw all members (edges)
for _, edge in df_edges.iterrows():
    v1 = _to_vertex_key(edge["V1"])
    v2 = _to_vertex_key(edge["V2"])

    if v1 not in vertex_lookup or v2 not in vertex_lookup:
        continue

    p1 = vertex_lookup[v1]
    p2 = vertex_lookup[v2]
    ax.plot3D(
        [p1["x"], p2["x"]],
        [p1["y"], p2["y"]],
        [p1["z"], p2["z"]],
        color="0.45",
        linewidth=0.8,
        alpha=0.75,
    )

# Draw nodes per layer for readability
df_top = df_vertices[df_vertices["layer"] == "top"]
df_bottom = df_vertices[df_vertices["layer"] == "bottom"]

ax.scatter3D(
    np.asarray(df_top["x"]),
    np.asarray(df_top["y"]),
    np.asarray(df_top["z"]),
    s=35,
    c="#1f77b4",
    label="top",
)
ax.scatter3D(
    np.asarray(df_bottom["x"]),
    np.asarray(df_bottom["y"]),
    np.asarray(df_bottom["z"]),
    s=35,
    c="#d62728",
    label="bottom",
)

ax.set_title("Random Design - 3D Geometry Preview")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_zlabel("z [m]", labelpad=14)
ax.legend(loc="upper right")
ax.set_box_aspect((1, 1, 0.45))

# Keep extra space on the right so the z label is fully visible.
fig.subplots_adjust(left=0.06, right=0.90, bottom=0.08, top=0.93)
plt.show()

## Feature extraction

In [ ]:
import pandas as pd
from reconstruction import extract_beam_properties

# --- UITVOEREN VAN STAP 3 ---
print("Stap 3: Staaflengtes en profielen berekenen...")

# We gebruiken de tabellen uit de VORIGE cel!
df_slots = extract_beam_properties(df_vertices, df_edges)

print(f"✅ Succes! We hebben de eigenschappen van {len(df_slots)} balken berekend.")
print("\nDit is de input (Slots) voor je Cost Matrix (eerste 10 balken):")
print(df_slots.head(10))

# COST MATRIX

We now use one universal assignment formula for both reclaimed and new timber in the same cost matrix:

$$C_{i,j} = E_{embodied,i} + E_{prep,i} + E_{trans,i,j} + E_{waste,i,j} + E_{saw,i,j}$$

With:
- $E_{embodied,i} = V_{stock,i} \cdot M_{material,i}$ (from `ECC` in stock data)
- $E_{prep,i} = V_{stock,i} \cdot E_{prep}$
- $E_{trans,i,j} = \left(\frac{(V_{req}+V_{over})\cdot\rho}{1000}\right)\cdot D\cdot F_{trans}$
- $E_{waste,i,j} = V_{waste} \cdot E_{EoL}$
- $E_{saw,i,j} = P_{cut}$ when cutting is required, else 0

Interpretation by timber type:
- New timber: typically high `ECC`, usually low/zero preparation factor
- Reclaimed timber: `ECC` near 0, non-zero preparation factor

## Controle

In [ ]:
from cost_calculation import prepare_stock_cost_inputs

# Strict validation + resolved inputs used by build_cost_matrix
df_stock_prepared = prepare_stock_cost_inputs(df_input_stock) # type: ignore

print("\n✅ Stock input gevalideerd voor de universele cost-formule.")
print("Preview van resolved kolommen:")
display(
    df_stock_prepared[[
        'Member_ID',
        'State',
        'ECC',
        'Bewerkingsfactor',
        'PreparationFactor_Resolved',
        'mean_density',
        'Transport_Dist',
        'Emmisiefactor'
    ]].head(10)
 )

## Building of cost matrix

In [ ]:
# Print even ter controle hoeveel balken erin zitten
print(f"Hout-inventaris succesvol ingeladen: {len(df_input_stock)} balken beschikbaar.")

# --- Controle van de 
# Bepaal automatisch twee balken om te onderzoeken (1x RS en 1x NS)
rs_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('RS')]['Member_ID'].tolist()
ns_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('NS')]['Member_ID'].tolist()

balk_onderzoek = []
if rs_balken: balk_onderzoek.append(rs_balken[0]) # Pak de eerste Reclaimed balk
if ns_balken: balk_onderzoek.append(ns_balken[0]) # Pak de eerste New balk

print(f"🔍 Diepte-analyse gestart voor: {balk_onderzoek}")

In [ ]:
import numpy as np
import pandas as pd
import config
from cost_calculation import build_cost_matrix

# Diepte-analyse instellingen
target_slot_for_analysis = 'e7'
all_stock_ids = df_input_stock['Member_ID'].dropna().astype(str).tolist()

print(f"🔍 Diepte-analyse gestart voor slot: {target_slot_for_analysis}")
print(f"📦 Aantal stock items in analyse: {len(all_stock_ids)}")

# Bouw de matrix en log alle stock items (voor alle slots)
cost_matrix, verrijkte_stock, df_logs = build_cost_matrix(
    df_slots,
    df_input_stock,
    target_stock_ids=all_stock_ids
)

# 3. Presentatie van het resultaat
# Maak er een DataFrame van, puur om het netjes te kunnen printen.
df_cost_matrix_display = pd.DataFrame(
    cost_matrix,
    index=[f"{row['edge_id']} ({row['Length_Req']:.0f}mm)" for _, row in df_slots.iterrows()],
    columns=verrijkte_stock['Member_ID'].tolist()
)

df_cost_matrix_display.to_csv(config.EXPORT_PATH / 'final_cost_matrix.csv', index=True)

# --- PRESENTEER DE DIEPTE-ANALYSE VOOR EEN SPECIFIEKE SLOT-ID ---
print("\n" + "=" * 80)
print(f"🔬 DIEPTE-ANALYSE: ALLE FACTOREN VOOR SLOT {target_slot_for_analysis}")
print("=" * 80)

if not df_logs.empty:
    df_logs = df_logs.sort_values(by=['Slot_ID', 'Stock_ID']).reset_index(drop=True)
    slot_mask = df_logs['Slot_ID'].astype(str).str.strip().str.lower() == target_slot_for_analysis.lower()
    df_logs_slot = df_logs.loc[slot_mask].copy()

    # Zorg dat ALLE stock-items zichtbaar zijn voor deze slot, ook als er iets ontbreekt in logs
    df_all_stock = pd.DataFrame({'Stock_ID': all_stock_ids}).drop_duplicates()
    df_logs_slot = df_all_stock.merge(df_logs_slot, on='Stock_ID', how='left')

    # Vul lege velden op zodat ontbrekende logregels zichtbaar worden
    if 'Slot_ID' in df_logs_slot.columns:
        df_logs_slot['Slot_ID'] = df_logs_slot['Slot_ID'].fillna(target_slot_for_analysis)
    if 'Status' in df_logs_slot.columns:
        df_logs_slot['Status'] = df_logs_slot['Status'].fillna('⚠️ ontbreekt in log')

    # Toon eerst alle RS-items expliciet
    rs_mask = df_logs_slot['Stock_ID'].astype(str).str.contains('RS', case=False, na=False)
    df_logs_slot_rs = df_logs_slot.loc[rs_mask].copy()

    print(f"\nAantal RS-items voor {target_slot_for_analysis}: {len(df_logs_slot_rs)}")
    if not df_logs_slot_rs.empty:
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
            display(df_logs_slot_rs.fillna('-').reset_index(drop=True))
    else:
        print("Geen RS-items gevonden in de input stock lijst.")

    # Toon daarna de volledige tabel (NS + RS)
    print("\nVolledige lijst (NS + RS):")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
        display(df_logs_slot.fillna('-').reset_index(drop=True))

    # Exporteer volledige diepte-analyse voor e0 zodat niets verloren gaat in output-weergave
    analysis_export_path = config.EXPORT_PATH / f"diepte_analyse_{target_slot_for_analysis}.csv"
    df_logs_slot.to_csv(analysis_export_path, index=False)
    print(f"\n💾 Diepte-analyse geëxporteerd naar: {analysis_export_path}")
else:
    print("Geen logboek data gevonden.")

print("=" * 80)

print("\nPreview Cost Matrix (eerste 8 staven, eerste 5 inventaris-balken):")
print("(Let op: 'inf' betekent dat de inventaris-balk te klein is en wordt uitgesloten)\n")
print("=" * 80)
print(df_cost_matrix_display.iloc[:8, :5].round(2))

# MATCHING ALGORITHM / ILP

In [ ]:
import pulp
import numpy as np
import pandas as pd

# ==========================================
# CEL 4: TOEPASSING VAN HET OPTIMALISATIE ALGORITME (MILP)
# ==========================================
print("Start MILP Optimizer voor definitieve toewijzing...")

# 1. DATA KOPPELEN VANUIT VORIGE SCRIPTS
# Haal de namen op uit de DataFrames van de vorige cellen
stock_items = verrijkte_stock['Member_ID'].tolist()
construction_slots = df_slots['edge_id'].tolist()

# Dynamische categorisatie (kijkt naar 'NS' en 'RS' in de Member_ID)
new_items = [item for item in stock_items if 'NS' in item]
reclaimed_items = [item for item in stock_items if 'RS' in item]

print(f"📊 Inventaris: {len(reclaimed_items)} Reclaimed, {len(new_items)} New elementen.")
print(f"📐 Constructie vereist: {len(construction_slots)} staven.")

# 2. FILTEREN VAN DE COST MATRIX (Sparse Matrix Maken)
# We maken alleen combinaties aan die fysiek passen (waar de cost NIET oneindig is)
valid_matches = []
costs = {}

for i, slot_id in enumerate(construction_slots):
    for j, stock_id in enumerate(stock_items):
        cost = cost_matrix[i, j]
        # Als de cost niet 'inf' is, is het een geldige match!
        if cost != np.inf:
            valid_matches.append((stock_id, slot_id))
            costs[(stock_id, slot_id)] = cost

print(f"⚡ Aantal geldige combinaties voor de solver gereduceerd tot: {len(valid_matches)}")

# 3. HET MODEL OPZETTEN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 4. DECISION VARIABLES
# We maken uitsluitend 'knoppen' aan voor de combinaties die kunnen passen
x = pulp.LpVariable.dicts("Match", valid_matches, 0, 1, pulp.LpBinary)

# 5. OBJECTIVE FUNCTION (Doel: Minimaliseer de totale CO2 penalty)
prob += pulp.lpSum([x[match] * costs[match] for match in valid_matches])

# 6. CONSTRAINTS (De Regels)

# Regel A: Elke staaf in de constructie MOET precies 1 balk toegewezen krijgen
for slot_id in construction_slots:
    # Zoek alle balken die überhaupt passen op dit slot
    valid_stock_for_slot = [stock_id for (stock_id, s_id) in valid_matches if s_id == slot_id]

    if not valid_stock_for_slot:
        print(f"⚠️ FATALE FOUT: Slot {slot_id} heeft GEEN ENKELE fysiek passende balk in de hele voorraad!")
    else:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for stock_id in valid_stock_for_slot]) == 1

# Regel B: Reclaimed hout is uniek (Max 1x gebruiken)
for stock_id in reclaimed_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= 1

# Regel C: Nieuw hout mag oneindig gebruikt worden (Limiet = aantal benodigde staven)
for stock_id in new_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= len(construction_slots)

# ==========================================
# 7. OPLOSSEN EN RESULTATEN
# ==========================================
prob.solve()

print("\n" + "="*50)
print(f"STATUS OPLOSSING: {pulp.LpStatus[prob.status]}")
print("="*50)

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)

    # Sla de winnende combinaties op in een overzichtelijke lijst
    results = []
    for j in construction_slots:
        for i in stock_items:
            match = (i, j)
            if match in x and x[match].varValue == 1:
                # Gebruik dezelfde kolomnamen als in de merge-stap
                results.append({'edge_id': j, 'assigned_timber': i, 'CO2_Penalty': round(costs[match], 2)})

    df_results = pd.DataFrame(results)

    print(f"\n✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van {total_cost:.2f} kg!")

    # ==========================================
    # 8. MERGEN MET DE ORIGINELE EDGE MATRIX
    # ==========================================
    # We gebruiken pd.merge om de nieuwe 'assigned_timber' kolom naast je bestaande V1/V2 kolommen te plakken
    df_final_edges = pd.merge(df_slots, df_results[['edge_id', 'assigned_timber']], on='edge_id', how='left')

    print("\nDefinitieve Hout-Toewijzing:")
    print("-" * 50)
    print(df_results.to_string(index=False))

    # Optioneel: Exporteer dit naar CSV om in Grasshopper te kleuren
    # df_final_edges.to_csv('definitieve_toewijzing.csv', index=False)

else:
    print("❌ Het algoritme kon geen oplossing vinden. Waarschijnlijk is je voorraad niet toereikend voor de opgevraagde constructie.")

# EXPORT

This is where the best parameters of the sctructure are exported, this is done in a format that can be used by grasshopper script to translate it into geometry

In [ ]:
# @title Export geometry + assigned stock
import pandas as pd
import config

# ==========================================
# 8. EXPORT VAN GEOMETRIE + MATCHING RESULTATEN
# ==========================================
# Parameters/keuzes
EXPORT_PREFIX = "optimum"              # bv. "optimum", "run01", ...
EXPORT_UNASSIGNED_AS = "UNASSIGNED"    # label voor edges zonder match
DROP_UNASSIGNED_EDGES = False           # True = alleen gematchte edges exporteren

print("\n📦 Exporteren van vertices en edges met toegewezen stock...")

# --- 1) Validatie van vereiste data uit eerdere cellen ---
required_vars = ["df_vertices", "df_edges", "df_results"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise ValueError(
        "Deze variabelen ontbreken uit eerdere cellen: " + ", ".join(missing) + 
        ". Run eerst geometry, matching en ILP-cellen."
    )

# --- 2) Maak edge export met assigned_timber ---
df_edges_export = df_edges.copy()
df_edges_export = pd.merge(
    df_edges_export,
    df_results[["edge_id", "assigned_timber", "CO2_Penalty"]],
    on="edge_id",
    how="left"
 )
df_edges_export["assigned_timber"] = df_edges_export["assigned_timber"].fillna(EXPORT_UNASSIGNED_AS)
df_edges_export["CO2_Penalty"] = df_edges_export["CO2_Penalty"].fillna(0)

if DROP_UNASSIGNED_EDGES:
    df_edges_export = df_edges_export[df_edges_export["assigned_timber"] != EXPORT_UNASSIGNED_AS].copy()

# --- 3) Bestandspaden opbouwen ---
pad_vertices = config.EXPORT_PATH / f"{EXPORT_PREFIX}_vertices.csv"
pad_edges = config.EXPORT_PATH / f"{EXPORT_PREFIX}_edges_with_stock.csv"

# --- 4) Exporteren ---
df_vertices.to_csv(pad_vertices, index=False)
df_edges_export.to_csv(pad_edges, index=False)

# --- 5) Samenvatting ---
n_total = len(df_edges_export)
n_matched = int((df_edges_export["assigned_timber"] != EXPORT_UNASSIGNED_AS).sum())
print(f"✅ Vertices geëxporteerd: {len(df_vertices)} -> {pad_vertices}")
print(f"✅ Edges geëxporteerd: {n_total} (matched: {n_matched}) -> {pad_edges}")

print("\nPreview edges + assigned_timber:")
display(df_edges_export.head(10))